In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.metrics import precision_score,accuracy_score,roc_auc_score
from sklearn.metrics import balanced_accuracy_score
from tqdm import tqdm
import pickle
def makefile(what,filename):
    with open(filename,"wb") as f3:
        pickle.dump(what,f3)

def readfile(filename):
    with open(filename,"rb") as f4:
        ans=pickle.load(f4)
    return ans
import random
from sklearn.datasets import fetch_kddcup99
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA as pca

In [2]:
file=fetch_kddcup99()
data=file["data"]
X=np.concatenate([data[:,0:1],data[:,4:]],axis=1).astype("float")
target=file["target"]
for word in target:
    break
Y=np.zeros(len(target))
Y[:]=-1
Y[target==word]=1
e=0
#X=fetch_dataset[word]["data"]
#Y=fetch_dataset[word]["target"]
print(word,":",len(X[Y==1]),":",len(X[Y==-1]),":",len(X[0]))
X=(X-X.min(axis=0))/(X.max(axis=0)-X.min(axis=0))
X[pd.isnull(X)]=0
#random.seed(0)
d=len(X[0])
split=0.01
random_value=1
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=split, random_state=e,stratify=Y)
positive=X_train[y_train==1]
negative=X_train[y_train==-1]

b'normal.' : 97278 : 396743 : 38


/tmp/ipykernel_1111358/2933721577.py:14: RuntimeWarning: invalid value encountered in divide
  X=(X-X.min(axis=0))/(X.max(axis=0)-X.min(axis=0))


In [3]:
%%time
models=[]
#score_max=[]
#score_min=[]
for n in range(100):
    OCSVM=OneClassSVM()
    OCSVM.fit(positive[n*963:(n+1)*963])
    models.append(OCSVM)


for n in range(100):
    LOF=LocalOutlierFactor(novelty=True)
    LOF.fit(positive[n*963:(n+1)*963])
    models.append(LOF)

for n in range(100):
    IF=IsolationForest()
    IF.fit(positive[n*963:(n+1)*963])
    models.append(IF)

for n in range(100):
    GMM=GaussianMixture(n_components=1)
    GMM.fit(positive[n*963:(n+1)*963])
    models.append(GMM)

CPU times: user 16.2 s, sys: 108 ms, total: 16.3 s
Wall time: 11.1 s


In [4]:
%%time
#score_max=[]
#score_min=[]
for n in range(100):
    OCSVM=OneClassSVM()
    OCSVM.fit(negative[n*3927:(n+1)*3927])
    models.append(OCSVM)

for n in range(100):
    LOF=LocalOutlierFactor(novelty=True)
    LOF.fit(negative[n*3927:(n+1)*3927])
    models.append(LOF)

for n in range(100):
    IF=IsolationForest()
    IF.fit(negative[n*3927:(n+1)*3927])
    models.append(IF)

for n in range(100):
    GMM=GaussianMixture(n_components=1)
    GMM.fit(negative[n*3927:(n+1)*3927])
    models.append(GMM)


CPU times: user 49.7 s, sys: 144 ms, total: 49.9 s
Wall time: 36.2 s


In [7]:
rankings=scores.argsort(axis=1).argsort(axis=1)
similarity_ovo=[]
n=scores.shape[1]
for rank1 in tqdm(rankings):
    similarity=[]
    for rank2 in rankings:
        #Spearman correlation
        similarity.append(1-(((abs(rank1-rank2)**2).sum()*6)/(n**3-n)))
    similarity_ovo.append(similarity)
similarity_ovo=np.array(similarity_ovo)

100%|████████████████████████████████████████| 800/800 [00:05<00:00, 140.23it/s]


In [8]:
similarity_ovo

array([[ 1.        ,  0.96984805,  0.85123174, ..., -0.14628937,
        -0.13833571, -0.15066668],
       [ 0.96984805,  1.        ,  0.85368922, ..., -0.14445228,
        -0.13622897, -0.14846618],
       [ 0.85123174,  0.85368922,  1.        , ..., -0.20015831,
        -0.19189536, -0.20421172],
       ...,
       [-0.14628937, -0.14445228, -0.20015831, ...,  1.        ,
         0.99713243,  0.99768602],
       [-0.13833571, -0.13622897, -0.19189536, ...,  0.99713243,
         1.        ,  0.99724405],
       [-0.15066668, -0.14846618, -0.20421172, ...,  0.99768602,
         0.99724405,  1.        ]])

In [10]:
#Normal vs abnormal
#leave-one-out cross-validation.
classes=np.zeros(len(rankings))
for i in range(2):
    classes[i*400:(i+1)*400]=i
y_pred=classes[similarity_ovo.argsort(axis=1)[:,-2]] #-1 returns the index of the model itself, so select -2.
accuracy_score(classes,y_pred)

1.0

In [11]:
#two-fold cross-validation
rank_train, rank_test, classes_train, classes_test= train_test_split(rankings,classes, test_size=0.5,random_state=0,stratify=classes)
similarity_ovo=[]
for rank1 in tqdm(rank_test):
    similarity=[]
    for rank2 in rank_train:
        similarity.append(1-(((abs(rank1-rank2)**2).sum()*6)/(n**3-n)))
    similarity_ovo.append(similarity)
similarity_ovo=np.array(similarity_ovo)
y_pred=classes_train[similarity_ovo.argmax(axis=1)]
print(accuracy_score(classes_test,y_pred))

similarity_ovo=[]
for rank1 in tqdm(rank_train):
    similarity=[]
    for rank2 in rank_test:
        similarity.append(1-(((abs(rank1-rank2)**2).sum()*6)/(n**3-n)))
    similarity_ovo.append(similarity)
similarity_ovo=np.array(similarity_ovo)
y_pred=classes_test[similarity_ovo.argmax(axis=1)]
print(accuracy_score(classes_train,y_pred))

100%|████████████████████████████████████████| 400/400 [00:01<00:00, 289.48it/s]


1.0


100%|████████████████████████████████████████| 400/400 [00:01<00:00, 295.90it/s]

1.0
